In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
import os
from time import sleep
from pathlib import Path
from urllib.parse import quote, unquote, urlparse, parse_qs
import numpy as np
from openpyxl import load_workbook

from birddog.wiki import (
    get_title,
    WIKI_NAMESPACE,
    download_thumbnail,
    )

from birddog.nocodb import (
    process_archive_sheet,
    process_fond_sheet,
    process_opus_sheet,
    process_worksheets,
    NocoDBClient,
    )

2025-12-04 15:51:38,755 [INFO] Translation is enabled. Using GCP translator
2025-12-04 15:51:38,756 [INFO] Using Google Cloud translation API
2025-12-04 15:51:38,756 [INFO] GoogleCloudTranslator using REST API


In [3]:
root = "./var/WAT"

In [4]:
def list_files(directory: str, suffix: str, prefix=None) -> list[Path]:
    """Recursively list only files in `directory` with the given `suffix`,
    skipping any whose name starts with '~'.
    """
    return [
        p for p in Path(directory).rglob(f"*{suffix}")
        if p.is_file() and not p.name.startswith("~")
        if not prefix or p.name.startswith(prefix)
    ]

In [23]:
list_files(root, "xlsx")

[PosixPath('var/WAT/DARO (Rivne)/DARO-D-wiki-20250915.xlsx'),
 PosixPath('var/WAT/DARO (Rivne)/DARO-R-wiki-20250728.xlsx'),
 PosixPath('var/WAT/DAVO (Volhynia)/DAVO-archives+AK-20250525b.xlsx'),
 PosixPath('var/WAT/DAVO (Volhynia)/DAVO-D-wiki-20250915.xlsx'),
 PosixPath('var/WAT/DAVO (Volhynia)/DAVO-R-wiki-20250915.xlsx'),
 PosixPath('var/WAT/DAHO (Kharkiv)/DAHO-archive-20250719.xlsx'),
 PosixPath('var/WAT/DAHO (Kharkiv)/DAHO-R-wiki-20250908.xlsx'),
 PosixPath('var/WAT/DAHO (Kharkiv)/DAHO-D-wiki-20250817a.xlsx'),
 PosixPath('var/WAT/DACHKO (Cherkassy)/DACHKO-R-wiki-20250904.xlsx'),
 PosixPath('var/WAT/DACHKO (Cherkassy)/DACHKO-D-wiki-20250904.xlsx'),
 PosixPath('var/WAT/DAHEO (Kherson)/DAHEO-R-archive-20250819.xlsx'),
 PosixPath('var/WAT/DAHEO (Kherson)/DAHEO-P-wiki-20250708.xlsx'),
 PosixPath('var/WAT/DAHEO (Kherson)/DAHEO-D-wiki-20250821.xlsx'),
 PosixPath('var/WAT/DAHEO (Kherson)/DAHEO-R-wiki-20250724.xlsx'),
 PosixPath('var/WAT/DAHEO (Kherson)/DAHEO-D-archive-20250707.xlsx'),
 Posi

In [6]:
#wb = load_workbook(list_files(root, "xlsx", prefix="DAHEO-D-wiki")[0])

In [7]:
client = NocoDBClient()

2025-12-04 15:51:52,835 [INFO] pages loaded (760 records)
2025-12-04 15:52:01,879 [INFO] documents loaded (464 records)


In [ ]:
#client.upsert_pages(process_worksheets(wb.worksheets[1:4]))

In [ ]:
#client.upsert_pages(process_worksheets(wb.worksheets[4:6]))

In [ ]:
#client.upsert_pages(process_worksheets(wb.worksheets[:1]))

In [ ]:
#client.upsert_pages(process_worksheets(wb.worksheets[6:]))

In [ ]:
#wb = load_workbook(list_files(root, "xlsx", prefix="DAHEO-P-wiki")[0])

In [ ]:
#wb.worksheets

In [ ]:
#client.upsert_pages(process_worksheets(wb.worksheets, page_table={}))

In [12]:
#wb = load_workbook(list_files(root, "xlsx", prefix="DAHEO-R-wiki")[0])

In [13]:
#client.upsert_pages(process_worksheets(wb.worksheets, page_table={}))

In [ ]:
docs = client.list_records("documents")

In [ ]:
doc_links = [item["link"] for item in docs if not item["doc_image"]]

In [ ]:
len(doc_links)

In [ ]:
#download_thumbnail(doc_links[1], out_dir="./var/thumbs")

In [ ]:
result = client.upload_attachment("./var/thumbs/ДАХеО_Фонд_1_Описи_1,_2.pdf.p1.w640.jpg")

In [ ]:
docs[1]

In [ ]:
download_result = download_thumbnail(docs[1]["link"], out_dir="./var/thumbs")
client.set_attachment("documents", docs[1]["link"], "doc_image", download_result["saved_path"])

In [ ]:
for doc in docs:
    if not doc["doc_image"]:
        print(f"Trying {doc["title"]}...")
        try:
            download_result = download_thumbnail(doc["link"], out_dir="./var/thumbs")
            client.set_attachment("documents", doc["link"], "doc_image", download_result["saved_path"])
        except Exception as e:
            print(f"failed: {e}")

In [14]:
list_files(root, "xlsx", prefix="DAHO")

[PosixPath('var/WAT/DAHO (Kharkiv)/DAHO-archive-20250719.xlsx'),
 PosixPath('var/WAT/DAHO (Kharkiv)/DAHO-R-wiki-20250908.xlsx'),
 PosixPath('var/WAT/DAHO (Kharkiv)/DAHO-D-wiki-20250817a.xlsx')]

In [15]:
archives = ["DAHO-D", "DAHO-R"]

In [16]:
for archive in archives:
    file = list_files(root, "xlsx", prefix=f"{archive}-wiki")
    if file:
        print(file[0])

var/WAT/DAHO (Kharkiv)/DAHO-D-wiki-20250817a.xlsx
var/WAT/DAHO (Kharkiv)/DAHO-R-wiki-20250908.xlsx


In [24]:
def load_workbooks(prefix, only_wiki=True):
    files = list_files(root, "xlsx", prefix=prefix)
    return [load_workbook(str(file)) 
            for file in list_files(root, "xlsx", prefix=prefix)
            if "wiki" in str(file) or not only_wiki]

In [21]:
def upsert_archive(archive):
    wbs = load_workbooks(archive)
    if wbs:
        client.upsert_pages(process_worksheets(wbs[0].worksheets, page_table={}))

In [18]:
upsert_archive(archives[0])

upserting spreadsheet: var/WAT/DAHO (Kharkiv)/DAHO-D-wiki-20250817a.xlsx
2025-12-04 16:14:25,559 [INFO] processing worksheet: DAHO D wiki fund list
2025-12-04 16:14:25,572 [INFO] processing worksheet: fund 3
2025-12-04 16:14:25,572 [INFO] processing worksheet: DAHO 3-287
2025-12-04 16:14:25,573 [INFO] processing worksheet: fund 4
2025-12-04 16:14:25,573 [INFO] processing worksheet: DAHO 4-160
2025-12-04 16:14:25,574 [INFO] processing worksheet: DAHO 4-163
2025-12-04 16:14:25,575 [INFO] processing worksheet: DAHO 4-166
2025-12-04 16:14:25,575 [INFO] processing worksheet: DAHO 4-177
2025-12-04 16:14:25,576 [INFO] processing worksheet: DAHO 4-187
2025-12-04 16:14:25,577 [INFO] processing worksheet: fund 31
2025-12-04 16:14:25,577 [INFO] processing worksheet: DAHO 31-141
2025-12-04 16:14:25,586 [INFO] processing worksheet: fund 52
2025-12-04 16:14:25,586 [INFO] processing worksheet: DAHO 52-1
2025-12-04 16:14:25,587 [INFO] processing worksheet: DAHO 52-2
2025-12-04 16:14:25,588 [INFO] proc

ValueError: Record must have title field

In [27]:
w = load_workbooks("DAHO")

In [28]:
w

In [31]:
pages=process_worksheets(w[1].worksheets, page_table={})

2025-12-05 09:40:32,416 [INFO] processing worksheet: DAHO D wiki fund list
2025-12-05 09:40:32,448 [INFO] processing worksheet: fund 3
2025-12-05 09:40:32,449 [INFO] processing worksheet: DAHO 3-287
2025-12-05 09:40:32,450 [INFO] processing worksheet: fund 4
2025-12-05 09:40:32,451 [INFO] processing worksheet: DAHO 4-160
2025-12-05 09:40:32,451 [INFO] processing worksheet: DAHO 4-163
2025-12-05 09:40:32,453 [INFO] processing worksheet: DAHO 4-166
2025-12-05 09:40:32,454 [INFO] processing worksheet: DAHO 4-177
2025-12-05 09:40:32,455 [INFO] processing worksheet: DAHO 4-187
2025-12-05 09:40:32,456 [INFO] processing worksheet: fund 31
2025-12-05 09:40:32,457 [INFO] processing worksheet: DAHO 31-141
2025-12-05 09:40:32,468 [INFO] processing worksheet: fund 52
2025-12-05 09:40:32,468 [INFO] processing worksheet: DAHO 52-1
2025-12-05 09:40:32,469 [INFO] processing worksheet: DAHO 52-2
2025-12-05 09:40:32,470 [INFO] processing worksheet: fund 179
2025-12-05 09:40:32,470 [INFO] processing work

In [36]:
print([p["label"] for p in pages.values()])

['DAHO-D', 'General page content', '3', '4', '6', '7', '8', '9', '11', '12', '14', '15', '16', '17', '18', '19', '20', '21', '23', '24', '25', '26', '27', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '40', '41', '42', '43', '44', '45', '46', '49', '50', '51', '52', '54', '55', '56', '57', '58', '60', '61', '62', '63', '64', '65', '67', '69', '71', '72', '73', '74', '77', '78', '79', '80', '82', '83', '84', '86', '87', '88', '89', '90', '91', '92', '93', '94', '98', '99', '101', '102', '103', '104', '105', '106', '107', '108', '113', '114', '116', '117', '118', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '135', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '160', '161', '162', '163', '164', '165', '167', '168', '170', '172', '173', '174', '175', '176', '177', '178', '179', '180', '181', '182', '183', '184', '185'